In [1]:
import enum
import pyomo.environ as pyo 

SOLVER = pyo.SolverFactory('appsi_highs')
assert SOLVER.available(), "HiGHS (appsi_highs) solver is not available."
print("Solver ready:", SOLVER)


Solver ready: <pyomo.contrib.appsi.base.SolverFactoryClass.register.<locals>.decorator.<locals>.LegacySolver object at 0x000002BCAC849160>


# Problem 1

In [ ]:
# Persons
"""
1 => Carl
2 => Chris
3 => David
4 => Tony
5 => Ken
"""
class Swimmer(enum.Enum):
    CARL = 1
    CHRIS = 2
    DAVID = 3
    TONY = 4
    KEN = 5

# Strokes
"""
1 => Backstroke
2 => Breaststroke
3 => Butterfly
4 => Freestyle
"""
class Stroke(enum.Enum):
    BACKSTROKE = 1
    BREASTSTROKE = 2
    BUTTERFLY = 3
    FREESTYLE = 4

# (Swimmer, Stroke) -> Time 
times = {
    (Swimmer.CARL, Stroke.BACKSTROKE): 37.7,
    (Swimmer.CARL, Stroke.BREASTSTROKE): 43.4,
    (Swimmer.CARL, Stroke.BUTTERFLY): 33.3,
    (Swimmer.CARL, Stroke.FREESTYLE): 29.2,

    (Swimmer.CHRIS, Stroke.BACKSTROKE): 32.9,
    (Swimmer.CHRIS, Stroke.BREASTSTROKE): 33.1,
    (Swimmer.CHRIS, Stroke.BUTTERFLY): 28.5,
    (Swimmer.CHRIS, Stroke.FREESTYLE): 26.4,

    (Swimmer.DAVID, Stroke.BACKSTROKE): 33.8,
    (Swimmer.DAVID, Stroke.BREASTSTROKE): 42.2,
    (Swimmer.DAVID, Stroke.BUTTERFLY): 38.9,
    (Swimmer.DAVID, Stroke.FREESTYLE): 29.6,

    (Swimmer.TONY, Stroke.BACKSTROKE): 37.0,
    (Swimmer.TONY, Stroke.BREASTSTROKE): 34.7,
    (Swimmer.TONY, Stroke.BUTTERFLY): 30.4,
    (Swimmer.TONY, Stroke.FREESTYLE): 28.5,

    (Swimmer.KEN, Stroke.BACKSTROKE): 35.4,
    (Swimmer.KEN, Stroke.BREASTSTROKE): 41.8,
    (Swimmer.KEN, Stroke.BUTTERFLY): 33.6,
    (Swimmer.KEN, Stroke.FREESTYLE): 31.1,
}


model = pyo.ConcreteModel()
model.SWIMMERS = pyo.Set(initialize=Swimmer)
model.STROKES = pyo.Set(initialize=Stroke)


model.x = pyo.Var(model.SWIMMERS, model.STROKES, domain=pyo.Binary)

model.obj = pyo.Objective(expr=sum(times[i, j] * model.x[i, j] for i in model.SWIMMERS for j in model.STROKES), sense=pyo.minimize)

def swimmer_assignment_rule(model, i):
    return sum(model.x[i, j] for j in model.STROKES) <= 1 
model.swimmer_assignment = pyo.Constraint(model.SWIMMERS, rule=swimmer_assignment_rule)

def stroke_assignment_rule(model, j):
    return sum(model.x[i,j] for i in model.SWIMMERS) == 1
model.stroke_assignment = pyo.Constraint(model.STROKES, rule=stroke_assignment_rule)

results = SOLVER.solve(model)
print("Status:", results.solver.status)

print("Optimal Time:", pyo.value(model.obj))
print("Assignments:")
for i in model.SWIMMERS:
    for j in model.STROKES:
        if pyo.value(model.x[i, j]) == 1:
            print(f"{i.name} assigned to {j.name}")


Status: ok
Optimal Time: 126.2
Assignments:
CARL assigned to FREESTYLE
CHRIS assigned to BUTTERFLY
DAVID assigned to BACKSTROKE
TONY assigned to BREASTSTROKE


\newpage

# Problem 2

In [ ]:
class Nodes(enum.Enum):
    origin = 0
    A = 1
    B = 2
    C = 3
    D = 4
    E = 5
    destination = 6

distances = {
    (Nodes.origin, Nodes.A) : 40, (Nodes.origin, Nodes.B) : 60, (Nodes.origin, Nodes.C) : 50,
    (Nodes.A, Nodes.B) : 10, (Nodes.A, Nodes.D) : 70,
    (Nodes.B, Nodes.C) : 20, (Nodes.B, Nodes.D) : 55, (Nodes.B, Nodes.E) : 40, 
    (Nodes.C, Nodes.E) : 50, 
    (Nodes.D, Nodes.destination) : 60, (Nodes.D, Nodes.E) : 10,
    (Nodes.E, Nodes.destination) : 80
}

b = {v : 0 for v in Nodes}
b [Nodes.origin] = 1
b [Nodes.destination] = -1

IN = {v : [] for v in Nodes}
OUT = {v : [] for v in Nodes}

for (i,j) in distances:
    OUT[i].append((i,j))
    IN[j].append((i,j))

model = pyo.ConcreteModel()

model.Nodes = pyo.Set(initialize=Nodes)
model.Edges = pyo.Set(initialize=distances.keys(), dimen=2)


model.distances = pyo.Param(model.Edges, initialize=distances)
model.b = pyo.Param(model.Nodes, initialize=b)


model.x = pyo.Var(model.Edges, domain=pyo.Binary)


model.obj = pyo.Objective(expr=sum(model.distances[i,j] * model.x[i,j] for (i,j) in model.Edges), sense = pyo.minimize)

model.IN = pyo.Set(model.Nodes, initialize = lambda m, v: IN[v], dimen = 2)
model.OUT = pyo.Set(model.Nodes, initialize = lambda m, v: OUT[v], dimen = 2)

def flow_conservation_rule(model, v):
    return sum(model.x[i,j] for (i,j) in model.OUT[v]) - sum(model.x[i,j] for (i,j) in model.IN[v]) == model.b[v]
model.flow_conservation = pyo.Constraint(model.Nodes, rule=flow_conservation_rule)

results = SOLVER.solve(model)
print("Status:", results.solver.status)
print("Optimal Distance:", pyo.value(model.obj))
print("Shortest path:")
for (i,j) in model.Edges:
    if pyo.value(model.x[i,j]) == 1:
        print(f"Travel from {i.name} to {j.name}")
        


Status: ok
Optimal Distance: 165.0
Shortest path:
Travel from origin to A
Travel from A to B
Travel from B to D
Travel from D to destination


In [83]:
class Nodes(enum.Enum):
    origin = 0
    A = 1
    B = 2
    C = 3
    D = 4
    E = 5
    destination = 6


flows = {
    (Nodes.origin, Nodes.A) : 40, (Nodes.origin, Nodes.B) : 60, (Nodes.origin, Nodes.C) : 50,
    (Nodes.A, Nodes.B) : 10, (Nodes.A, Nodes.D) : 70,
    (Nodes.B, Nodes.C) : 20, (Nodes.B, Nodes.D) : 55, (Nodes.B, Nodes.E) : 40, 
    (Nodes.C, Nodes.E) : 50, 
    (Nodes.D, Nodes.destination) : 60, (Nodes.D, Nodes.E) : 10,
    (Nodes.E, Nodes.destination) : 80,
    (Nodes.destination, Nodes.origin) : 0
}

IN = {v : [] for v in Nodes}
OUT = {v : [] for v in Nodes}

for (i,j) in flows:
    OUT[i].append((i,j))
    IN[j].append((i,j))

flow_model = pyo.ConcreteModel("max_flow")

flow_model.Nodes = pyo.Set(initialize=Nodes)
flow_model.Edges = pyo.Set(initialize=flows.keys(), dimen=2)
flow_model.max_flow = pyo.Param(flow_model.Edges, initialize = flows)


# the flow from each edge 
flow_model.x = pyo.Var(flow_model.Edges, domain=pyo.NonNegativeReals)

flow_model.obj = pyo.Objective(expr=flow_model.x[Nodes.destination, Nodes.origin], sense = pyo.maximize)

# nodes that enter and exit node i 
flow_model.IN = pyo.Set(flow_model.Nodes, initialize = lambda m, v: IN[v], dimen = 2)
flow_model.OUT = pyo.Set(flow_model.Nodes, initialize = lambda m, v: OUT[v], dimen = 2)

# make sure that the flow into a node is equal to the flow out of the node
def flow_conservation_rule(model, v):
    return sum(model.x[i,j] for (i,j) in model.OUT[v]) == sum(model.x[i,j] for (i,j) in model.IN[v]) 
flow_model.flow_conservation = pyo.Constraint(flow_model.Nodes, rule=flow_conservation_rule)

# make sure that the flow capacity is followed 
def flow_constraints(model, i,j):
    if i == Nodes.destination and j == Nodes.origin:
        return pyo.Constraint.Feasible
    return model.x[i,j] <= model.max_flow[i,j]
flow_model.capacity = pyo.Constraint(flow_model.Edges, rule = flow_constraints)

results = SOLVER.solve(flow_model)
print("Status:", results.solver.status)
print("Max Flow:", pyo.value(flow_model.obj))
print("_"*30)
print("Flow from each node: ")

for (i,j) in flow_model.Edges: 
    print(f"Flow from {i.name} to {j.name} is {flow_model.x[i,j].value}")


Status: ok
Max Flow: 140.0
______________________________
Flow from each node: 
Flow from origin to A is 40.0
Flow from origin to B is 50.0
Flow from origin to C is 50.0
Flow from A to B is 0.0
Flow from A to D is 40.0
Flow from B to C is 0.0
Flow from B to D is 20.0
Flow from B to E is 30.0
Flow from C to E is 50.0
Flow from D to destination is 60.0
Flow from D to E is 0.0
Flow from E to destination is 80.0
Flow from destination to origin is 140.0
